In [ ]:
# GEN AI - TEXT RPG: CURSE OF ASHENVEIL
# This notebook is a text-based RPG that uses GPT-2 to write
# scene descriptions on the fly. The actual game logic (combat,
# inventory, branching story) is all regular Python — GPT-2 just
# handles the flavour text so every playthrough feels a bit different.

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import random  # used for damage variance in combat

# -------------------------------------------------
# LOAD THE MODEL
# We're using GPT-2, a small but capable language model from OpenAI.
# AutoTokenizer converts text -> numbers the model understands.
# AutoModelForCausalLM is the actual neural network that predicts text.
# Both download automatically the first time you run this.
# -------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")


# -------------------------------------------------
# TEXT GENERATION
# This is the core AI function. We hand it a prompt (the start of
# a sentence) and it completes it. max_new_tokens controls how long
# the output can be. temperature controls creativity — higher = wilder.
# -------------------------------------------------
def generate_text(prompt, max_new_tokens=50):
    # tokenize the prompt into a tensor GPT-2 can read
    inputs = tokenizer(prompt, return_tensors="pt")

    # generate a continuation — do_sample=True means it picks words
    # probabilistically rather than always choosing the most likely one
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.9,   # tweak this to change the writing style
        top_k=50,          # only consider the top 50 most likely next words
        pad_token_id=tokenizer.eos_token_id,  # avoids a warning about padding
    )

    # decode the output back to a string, then strip the original prompt
    # so we only return the NEW text GPT-2 wrote
    full_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return full_text[len(prompt):].strip()


# -------------------------------------------------
# ITEM DEFINITIONS
# Each item has a type, an effect, and a value.
# type:   weapon | armor | consumable
# effect: what the item does when used or equipped
# value:  how much stat change or healing it provides
# -------------------------------------------------
ITEMS = {
    "Health Potion":   {"type": "consumable", "effect": "heal",    "value": 30, "desc": "Restores 30 HP"},
    "Iron Sword":      {"type": "weapon",     "effect": "attack",  "value": 10, "desc": "+10 attack power"},
    "Enchanted Blade": {"type": "weapon",     "effect": "attack",  "value": 20, "desc": "+20 attack power"},
    "Leather Armor":   {"type": "armor",      "effect": "defense", "value": 5,  "desc": "+5 defense"},
    "Steel Shield":    {"type": "armor",      "effect": "defense", "value": 10, "desc": "+10 defense"},
    "Smoke Bomb":      {"type": "consumable", "effect": "escape",  "value": 0,  "desc": "Escape any battle"},
    "Poison Dagger":   {"type": "weapon",     "effect": "poison",  "value": 8,  "desc": "Poisons enemy for 5 dmg/turn"},
}


# -------------------------------------------------
# PLAYER STATE
# One dictionary holds everything about the player.
# We modify this throughout the game as the player
# picks up items, takes damage, and earns gold.
# -------------------------------------------------
player = {
    "name":       "",
    "health":     100,
    "max_health": 100,
    "attack":     10,    # increases when a weapon is equipped
    "defense":    0,     # reduces incoming damage; increases with armor
    "inventory":  [],    # list of item name strings
    "gold":       0,
    "weapon":     None,  # only one weapon equipped at a time
    "armor":      None,  # only one armor piece at a time
    "poisoned":   False, # flag for poison status effect
}


# -------------------------------------------------
# UTILITY HELPERS
# Small reusable functions used throughout the game.
# -------------------------------------------------

def divider(char="=", length=60):
    # just prints a visual separator line in the output
    print(char * length)


def show_status():
    # prints the player's current stats so they always know
    # their HP, attack, defense, and what they're carrying
    print(f"\n[ {player['name']} | HP: {player['health']}/{player['max_health']} | "
          f"ATK: {player['attack']} | DEF: {player['defense']} | "
          f"Gold: {player['gold']} ]")
    if player['inventory']:
        print(f"Inventory: {', '.join(player['inventory'])}")
    else:
        print("Inventory: (empty)")


def pick_number(prompt, min_val, max_val):
    # keeps asking until the player types a valid number
    # handles non-numeric input gracefully with try/except
    while True:
        try:
            val = int(input(prompt))
            if min_val <= val <= max_val:
                return val
        except ValueError:
            pass
        print(f"Please enter a number between {min_val} and {max_val}.")


def pick_item(item_type=None):
    # shows the player their inventory (filtered by type if specified)
    # and returns the name of whichever item they choose
    available = [
        item for item in player["inventory"]
        if item_type is None or ITEMS[item]["type"] == item_type
    ]
    if not available:
        print("You have no usable items.")
        return None
    print("\nYour items:")
    for i, item in enumerate(available, 1):
        print(f"  {i}. {item} — {ITEMS[item]['desc']}")
    choice = pick_number("Choose an item (0 to cancel): ", 0, len(available))
    if choice == 0:
        return None
    return available[choice - 1]


def give_item(item_name):
    # adds an item to the inventory
    # if it's a weapon or armor and nothing is equipped yet,
    # auto-equip it and update the player's stats immediately
    player["inventory"].append(item_name)
    item = ITEMS[item_name]
    if item["type"] == "weapon" and player["weapon"] is None:
        player["weapon"] = item_name
        player["attack"] += item["value"]
        print(f"  You equip the {item_name}! Attack +{item['value']}.")
    elif item["type"] == "armor" and player["armor"] is None:
        player["armor"] = item_name
        player["defense"] += item["value"]
        print(f"  You equip the {item_name}! Defense +{item['value']}.")
    else:
        print(f"  You pick up: {item_name}.")


def use_item_in_combat(enemy):
    # called when the player chooses 'Use Item' during a fight
    # only consumables are usable mid-combat (weapons/armor are passive)
    # returns True (item used), False (no item/cancelled), or "escape"
    item_name = pick_item(item_type="consumable")
    if item_name is None:
        return False
    item = ITEMS[item_name]
    player["inventory"].remove(item_name)  # consume it
    if item["effect"] == "heal":
        # cap healing so we don't go over max HP
        healed = min(item["value"], player["max_health"] - player["health"])
        player["health"] += healed
        print(f"  You drink the {item_name} and recover {healed} HP!")
    elif item["effect"] == "escape":
        print("  You throw a Smoke Bomb and vanish into the shadows!")
        return "escape"
    return True


# -------------------------------------------------
# COMBAT SYSTEM
# Turn-based fighting. Each round the player picks:
#   1. Attack  — deal damage based on attack stat
#   2. Defend  — halve incoming damage this turn
#   3. Use Item — heal, escape, etc.
# Enemies just attack every turn (no fancy AI).
# Returns True if the player wins, False if they die/flee.
# -------------------------------------------------
def combat(enemy_name, enemy_health, enemy_attack, enemy_defense=0, reward_gold=0, reward_item=None):
    divider()
    print(f"COMBAT: {player['name']} vs {enemy_name}!")
    divider()

    # track whether the enemy has been poisoned by the Poison Dagger
    enemy_poisoned = False
    poison_turns   = 0

    while True:
        # show HP at the top of each round
        print(f"\n  {player['name']} HP: {player['health']}  |  {enemy_name} HP: {enemy_health}")
        divider("-")

        # --- player's turn ---
        print("What will you do?")
        print("  1. Attack")
        print("  2. Defend (halve incoming damage this turn)")
        print("  3. Use Item")
        action = pick_number("  ", 1, 3)

        defending = False  # reset each round

        if action == 1:
            # damage = player attack - enemy defense + some randomness
            # max(..., 1) makes sure we always deal at least 1 damage
            dmg = max(1, player["attack"] - enemy_defense + random.randint(-3, 3))
            enemy_health -= dmg
            print(f"  You strike {enemy_name} for {dmg} damage!")

            # Poison Dagger special — applies poison on first hit
            if player["weapon"] == "Poison Dagger" and not enemy_poisoned:
                enemy_poisoned = True
                poison_turns   = 3
                print(f"  {enemy_name} is POISONED! (5 dmg/turn for 3 turns)")

        elif action == 2:
            # set a flag — we'll use it when the enemy attacks below
            defending = True
            print("  You brace yourself, reducing incoming damage this turn.")

        elif action == 3:
            result = use_item_in_combat(enemy_health)
            if result == "escape":
                print("You flee the battle!")
                return False  # treat fleeing as a loss for story purposes

        # --- poison tick on the enemy ---
        if enemy_poisoned and poison_turns > 0:
            enemy_health -= 5
            poison_turns -= 1
            print(f"  Poison burns {enemy_name} for 5 damage! ({poison_turns} turns left)")
            if poison_turns == 0:
                enemy_poisoned = False

        # --- check if enemy is dead ---
        if enemy_health <= 0:
            print(f"\n  {enemy_name} has been defeated!")
            if reward_gold > 0:
                player["gold"] += reward_gold
                print(f"  You find {reward_gold} gold.")
            if reward_item:
                give_item(reward_item)
            return True

        # --- player poison tick (if they got poisoned somehow) ---
        if player["poisoned"]:
            player["health"] -= 5
            print("  You are poisoned! -5 HP.")

        # --- enemy attacks ---
        # same damage formula as the player's attack
        e_dmg = max(1, enemy_attack - player["defense"] + random.randint(-2, 2))
        if defending:
            e_dmg = e_dmg // 2  # integer division so it's always a whole number
        player["health"] -= e_dmg
        print(f"  {enemy_name} attacks you for {e_dmg} damage!")

        # --- check if player is dead ---
        if player["health"] <= 0:
            print(f"\n  You have been defeated by {enemy_name}...")
            return False


# -------------------------------------------------
# SCENE HELPER
# Every scene calls this. It:
#   1. Prints the player's stats
#   2. Runs the prompt through GPT-2 for flavour text
#   3. Shows the choices and waits for input
# -------------------------------------------------
def play_scene(scene_prompt, choices, show_stats=True):
    divider()
    if show_stats:
        show_status()
    # GPT-2 writes the scene description based on the prompt we give it
    description = generate_text(scene_prompt)
    print("\n" + description)
    divider()
    for i, choice in enumerate(choices, start=1):
        print(f"  {i}. {choice}")
    return pick_number("\nWhat do you do? ", 1, len(choices))


# ==========================================================
# THE STORY — 5 scenes, each returns a path string
# that shapes the prompts and options in the next scene.
# If a combat results in death, the scene returns None
# and the main loop ends the game.
# ==========================================================

def scene_village_gate():
    """Scene 1 — arriving at the ruined village."""
    choice = play_scene(
        f"{player['name']} stood at the crumbling gate of Ashenveil village. Smoke rose in the distance.",
        [
            "Enter the village and look for survivors",
            "Check the abandoned market stall first",
            "Sneak around the back alleys",
        ]
    )

    # each path gives a different starting item and a label
    # used to shape the next scene's GPT-2 prompt
    if choice == 1:
        print("\nA wounded villager presses an Iron Sword into your hands before collapsing.")
        give_item("Iron Sword")
        return "well"   # took the main road
    elif choice == 2:
        print("\nUnder the counter you find a dusty satchel — a Health Potion and some coin inside.")
        give_item("Health Potion")
        player["gold"] += 15
        print("  +15 gold.")
        return "market"  # went through the market
    else:
        print("\nHugging the shadows you spot a leather jerkin hanging from a hook — still good.")
        give_item("Leather Armor")
        return "alley"  # snuck through the back


def scene_dark_forest(path):
    """Scene 2 — navigating the forest outside the village."""
    # the GPT-2 prompt changes based on how the player left the village
    if path == "well":
        prompt = f"{player['name']} followed the old well-road into the whispering forest and heard"
    elif path == "market":
        prompt = f"{player['name']} took the merchant trail through the forest. The trees grew thicker and"
    else:
        prompt = f"{player['name']} crept through the dark forest. Branches clawed at their cloak as"

    choice = play_scene(
        prompt,
        [
            "Press deeper into the forest",
            "Investigate the strange glow ahead",
            "Set up camp and rest (+20 HP)",
        ]
    )

    if choice == 1:
        # wolf ambush — mid-difficulty fight
        print("\nA forest wolf leaps from the undergrowth!")
        alive = combat("Forest Wolf", enemy_health=35, enemy_attack=8,
                       reward_gold=10, reward_item="Health Potion")
        return "deep" if alive else None
    elif choice == 2:
        # witch shrine — costs gold but gives a strong weapon
        print("\nYou find a witch's shrine. She offers a weapon in exchange for coin.")
        if player["gold"] >= 10:
            player["gold"] -= 10
            give_item("Enchanted Blade")
        else:
            print("  You don't have enough gold (need 10). The shrine fades into the mist.")
        return "shrine"
    else:
        # rest to recover — safe choice but no loot
        healed = min(20, player["max_health"] - player["health"])
        player["health"] += healed
        print(f"\nYou rest by a small fire and recover {healed} HP.")
        return "rested"


def scene_ruined_castle(forest_path):
    """Scene 3 — approaching and entering Blackstone Castle."""
    choice = play_scene(
        f"{player['name']} emerged from the forest to find the ruined walls of Blackstone Castle. A drawbridge hung open.",
        [
            "Cross the drawbridge directly",
            "Search the outer walls for another way in",
            "Call out to whoever might be inside",
        ]
    )

    if choice == 1:
        # frontal approach = tough guard fight
        print("\nTwo castle guards block your path — they're not letting anyone through.")
        alive = combat("Castle Guard", enemy_health=50, enemy_attack=12,
                       enemy_defense=3, reward_gold=25)
        if not alive:
            return None
        print("\nThe second guard throws down his sword and runs. You walk in.")
        return "front"
    elif choice == 2:
        # sneaky route — free armor, no fight
        print("\nYou find a gap in the wall. Inside a chest: a Steel Shield, dusty but solid.")
        give_item("Steel Shield")
        return "side"
    else:
        # parley — optional fight with a knight; has better rewards
        print("\nA grizzled knight answers. He sizes you up and draws his sword.")
        alive = combat("Knight of Ash", enemy_health=40, enemy_attack=10,
                       enemy_defense=5, reward_gold=20, reward_item="Smoke Bomb")
        return "parley" if alive else None


def scene_throne_room(castle_path):
    """Scene 4 — confronting whatever sits on the throne."""
    choice = play_scene(
        f"{player['name']} entered the throne room. On the cracked throne sat a figure wrapped in shadow.",
        [
            "Challenge the figure to combat",
            "Try to negotiate",
            "Use the environment — pull a lever on the wall",
        ]
    )

    if choice == 1:
        # straight-up boss fight
        print("\nThe Shadow Warden rises, twin blades gleaming!")
        alive = combat("Shadow Warden", enemy_health=80, enemy_attack=18,
                       enemy_defense=6, reward_gold=50, reward_item="Poison Dagger")
        return "fought" if alive else None
    elif choice == 2:
        # negotiation — works only if you have village items as 'proof'
        print("\nThe figure listens. It demands proof that you came from the village.")
        if "Leather Armor" in player["inventory"] or "Iron Sword" in player["inventory"]:
            print("  You hold up the villager's items. The figure slowly sits back down.")
            player["gold"] += 30
            print("  +30 gold offered as a gesture of peace.")
            return "negotiated"
        else:
            # no proof — falls back to combat
            print("  You have nothing to show. The figure attacks anyway.")
            alive = combat("Shadow Warden", enemy_health=80, enemy_attack=18,
                           enemy_defense=6, reward_gold=50)
            return "fought" if alive else None
    else:
        # environmental kill — no combat needed, less gold
        print("\nThe lever triggers a trapdoor — the warden drops straight into the dungeon.")
        print("  The throne room is yours without a fight.")
        player["gold"] += 20
        return "tricked"


def scene_final(throne_path):
    """Scene 5 — the ending. GPT-2 writes the closing paragraph."""
    # the closing prompt is shaped by how the throne room went
    if throne_path == "fought":
        outcome_prompt = (
            f"{player['name']} stood over the fallen Shadow Warden, breathing hard. "
            f"The curse over Ashenveil shattered like glass, and"
        )
    elif throne_path == "negotiated":
        outcome_prompt = (
            f"{player['name']} and the Shadow Warden struck an uneasy truce. "
            f"As they walked out onto the battlements together,"
        )
    else:  # tricked
        outcome_prompt = (
            f"{player['name']} claimed the empty throne of Blackstone Castle. "
            f"From the high window the ruined village was visible far below, and"
        )

    divider()
    print("               THE FINAL CHAPTER")
    divider()
    show_status()  # show final stats before the closing paragraph

    # GPT-2 writes the last paragraph — longer token limit for a proper ending
    ending = generate_text(outcome_prompt, max_new_tokens=80)
    print("\n" + ending)
    divider()
    print(f"  Gold collected: {player['gold']}")
    print("\n--- THE END ---")


# -------------------------------------------------
# MAIN GAME LOOP
# Runs the scenes in order. Each scene returns either a
# path string (story continues) or None (player died).
# -------------------------------------------------
def start_game():
    divider("#")
    print("#        GPT-2 TEXT RPG — CURSE OF ASHENVEIL         #")
    divider("#")

    player["name"] = input("\nWhat is your hero's name? ").strip() or "Hero"
    print(f"\nWelcome, {player['name']}. The village of Ashenveil has gone dark.")
    print("You are the only one who can break the curse.\n")

    village_path = scene_village_gate()

    forest_path = scene_dark_forest(village_path)
    if forest_path is None:
        print("\nYour journey ends here. The forest claims another soul.")
        print("--- GAME OVER ---")
        return

    castle_path = scene_ruined_castle(forest_path)
    if castle_path is None:
        print("\nThe castle walls become your tomb.")
        print("--- GAME OVER ---")
        return

    throne_path = scene_throne_room(castle_path)
    if throne_path is None:
        print("\nDarkness swallows you in the throne room.")
        print("--- GAME OVER ---")
        return

    scene_final(throne_path)


# kick it off
start_game()
